# Datos Socio-Demográficos y Recursos de Salud por Condado en EE. UU. (2018-2019)

**Variable objetivo:** `Heart disease_number` — número absoluto de personas con enfermedad cardíaca por condado.

**Justificación:** Las enfermedades cardíacas son la principal causa de muerte en EE. UU. Analizamos cuántas personas por condado padecen esta enfermedad en función de los datos socio-demográficos y de recursos sanitarios disponibles.

## Paso 1: Carga del conjunto de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Opción A: desde el enlace del proyecto
# df = pd.read_csv('https://breathecode.herokuapp.com/asset/internal-link?id=733&path=demographic_health_data.csv')

# Opción B: repositorio público (mismo dataset)
df = pd.read_csv('https://raw.githubusercontent.com/4GeeksAcademy/regularized-linear-regression-project-tutorial/main/demographic_health_data.csv')

print(f'Dimensiones: {df.shape}')
df.head()

## Paso 2: Análisis Exploratorio de Datos (EDA)

### 2.1 Inspección general

In [ ]:
print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
print()
print('Tipos de datos:')
print(df.dtypes.value_counts())
print()
nulls = df.isnull().sum()
print('Valores nulos:')
print(nulls[nulls > 0] if nulls.any() else 'Sin valores nulos')

In [ ]:
# Estadísticas descriptivas de variables clave
key_cols = [
    'Heart disease_number', 'Obesity_number', 'diabetes_number',
    'COPD_number', 'PCTPOVALL_2018', 'Unemployment_rate_2018',
    'Median_Household_Income_2018', 'Active Physicians per 100000 Population 2018 (AAMC)'
]
df[key_cols].describe().round(2)

### 2.2 Selección de variables

El dataset tiene 108 columnas. Seleccionamos un subconjunto no redundante que incluye:
- **Variables demográficas** (conteos por grupo de edad y raza)
- **Variables socioeconómicas** (pobreza, desempleo, ingresos, educación)
- **Recursos sanitarios** (médicos, hospitales, camas UCI)
- **Otras condiciones de salud** (obesidad, diabetes, COPD — en número absoluto)
- **Variable objetivo:** `Heart disease_number`

In [ ]:
selected_cols = [
    # Demografía — conteos absolutos por franja de edad
    '0-9', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+',
    # Demografía — raza (conteos)
    'White-alone pop', 'Black-alone pop', 'Asian-alone pop',
    # Socioeconómico
    'PCTPOVALL_2018',
    'Unemployment_rate_2018',
    'Median_Household_Income_2018',
    'Percent of adults with less than a high school diploma 2014-18',
    "Percent of adults with a bachelor's degree or higher 2014-18",
    'R_birth_2018',
    'R_death_2018',
    'Urban_rural_code',
    # Recursos sanitarios
    'Active Physicians per 100000 Population 2018 (AAMC)',
    'Active Primary Care Physicians per 100000 Population 2018 (AAMC)',
    'Total nurse practitioners (2019)',
    'Total physician assistants (2019)',
    'Total Hospitals (2019)',
    'ICU Beds_x',
    # Otras condiciones (en número absoluto, consistente con la variable objetivo)
    'Obesity_number',
    'diabetes_number',
    'COPD_number',
    'CKD_number',
    # Variable objetivo
    'Heart disease_number'
]

df_sel = df[selected_cols].copy()
print(f'Variables seleccionadas: {df_sel.shape[1]} (de {df.shape[1]})')
df_sel.head()

### 2.3 Análisis de la variable objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df_sel['Heart disease_number'], kde=True, ax=axes[0], color='crimson')
axes[0].set_title('Distribución de Heart disease_number')
axes[0].set_xlabel('Número de casos')

sns.boxplot(y=df_sel['Heart disease_number'], ax=axes[1], color='crimson')
axes[1].set_title('Boxplot de Heart disease_number')

plt.tight_layout()
plt.show()

print(f"Media:   {df_sel['Heart disease_number'].mean():,.0f} casos")
print(f"Mediana: {df_sel['Heart disease_number'].median():,.0f} casos")
print(f"Std:     {df_sel['Heart disease_number'].std():,.0f} casos")
print(f"Min:     {df_sel['Heart disease_number'].min():,.0f} | Max: {df_sel['Heart disease_number'].max():,.0f}")

### 2.4 Correlaciones con la variable objetivo

In [ ]:
corr_target = df_sel.corr()['Heart disease_number'].drop('Heart disease_number').sort_values()

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#d73027' if v > 0 else '#4575b4' for v in corr_target.values]
corr_target.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación con Heart disease_number')
ax.set_xlabel('Coeficiente de Pearson')
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplots de las 6 variables con mayor correlación absoluta
top6 = corr_target.abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(top6):
    sns.regplot(data=df_sel, x=col, y='Heart disease_number',
                ax=axes[i], scatter_kws={'alpha': 0.3, 's': 10}, color='crimson')
    axes[i].set_title(f'{col[:35]}\n(r={corr_target[col]:.2f})')

plt.tight_layout()
plt.show()

### 2.5 Mapa de correlación general

In [ ]:
fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(df_sel.corr(), dtype=bool))
sns.heatmap(
    df_sel.corr(), mask=mask,
    annot=False, cmap='coolwarm', center=0,
    ax=ax, linewidths=0.3
)
ax.set_title('Mapa de correlación (triángulo inferior)', fontsize=14)
plt.tight_layout()
plt.show()

### 2.6 Tratamiento de outliers

In [ ]:
# Visualizamos la variable objetivo y variables de recursos
check_cols = ['Heart disease_number', 'Total Hospitals (2019)', 'ICU Beds_x',
              'Active Physicians per 100000 Population 2018 (AAMC)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, col in enumerate(check_cols):
    sns.boxplot(y=df_sel[col], ax=axes[i], color='steelblue')
    axes[i].set_title(col[:28], fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Eliminamos outliers extremos con IQR factor 3 en columnas de recursos
df_clean = df_sel.copy()

outlier_cols = [
    'Total Hospitals (2019)', 'ICU Beds_x',
    'Total nurse practitioners (2019)', 'Total physician assistants (2019)',
    'Active Physicians per 100000 Population 2018 (AAMC)'
]

for col in outlier_cols:
    Q1, Q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    before = df_clean.shape[0]
    df_clean = df_clean[
        (df_clean[col] >= Q1 - 3*IQR) & (df_clean[col] <= Q3 + 3*IQR)
    ]
    removed = before - df_clean.shape[0]
    if removed > 0:
        print(f'{col[:45]}: {removed} outliers eliminados')

print(f'\nFilas restantes: {df_clean.shape[0]}')

### 2.7 División train/test y normalización

In [ ]:
X = df_clean.drop('Heart disease_number', axis=1)
y = df_clean['Heart disease_number']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalización — necesaria para Lasso/Ridge (penaliza coeficientes)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape[0]} filas | Test: {X_test_sc.shape[0]} filas')
print(f'Variables predictoras: {X_train_sc.shape[1]}')

## Paso 3: Modelos de Regresión

### 3.1 Regresión Lineal base

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)

print('--- Regresión Lineal (base) ---')
print(f'MSE:  {mse_lr:,.2f}')
print(f'RMSE: {np.sqrt(mse_lr):,.2f}')
print(f'R²:   {r2_lr:.4f} ({r2_lr*100:.2f}%)')

### 3.2 Lasso con valores por defecto

In [ ]:
lasso_def = Lasso(random_state=42)  # alpha=1.0 por defecto
lasso_def.fit(X_train_sc, y_train)
y_pred_ldef = lasso_def.predict(X_test_sc)

mse_ldef = mean_squared_error(y_test, y_pred_ldef)
r2_ldef  = r2_score(y_test, y_pred_ldef)

print('--- Lasso (alpha=1.0, por defecto) ---')
print(f'MSE:  {mse_ldef:,.2f}')
print(f'RMSE: {np.sqrt(mse_ldef):,.2f}')
print(f'R²:   {r2_ldef:.4f} ({r2_ldef*100:.2f}%)')
print()
n_zero = (lasso_def.coef_ == 0).sum()
print(f'Variables reducidas a 0 por Lasso: {n_zero} de {X.shape[1]}')

### 3.3 Comparación LinearRegression vs Lasso (defecto)

In [ ]:
comp = pd.DataFrame({
    'Modelo': ['LinearRegression', 'Lasso (alpha=1.0)'],
    'MSE':  [mse_lr, mse_ldef],
    'RMSE': [np.sqrt(mse_lr), np.sqrt(mse_ldef)],
    'R²':   [r2_lr, r2_ldef]
})
print(comp.to_string(index=False))

### 3.4 Evolución del R² según el valor de alpha en Lasso

In [ ]:
alphas = list(np.arange(0.0, 0.1, 0.01)) + \
         list(np.arange(0.1, 1.0, 0.1)) + \
         list(np.arange(1, 21, 1))

r2_train_list, r2_test_list = [], []

for a in alphas:
    m = Lasso(alpha=max(a, 1e-6), max_iter=10000, random_state=42)
    m.fit(X_train_sc, y_train)
    r2_train_list.append(r2_score(y_train, m.predict(X_train_sc)))
    r2_test_list.append(r2_score(y_test,  m.predict(X_test_sc)))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(alphas, r2_train_list, label='R² Train', color='steelblue', marker='o', markersize=3)
ax.plot(alphas, r2_test_list,  label='R² Test',  color='crimson',   marker='o', markersize=3)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=1, label='alpha=1.0 (default)')
ax.set_xlabel('alpha')
ax.set_ylabel('R²')
ax.set_title('Evolución del R² de Lasso según alpha')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_idx   = np.argmax(r2_test_list)
best_alpha = alphas[best_idx]
print(f'Mejor alpha en test: {best_alpha} → R² = {r2_test_list[best_idx]:.4f}')

## Paso 4: Optimización del modelo Lasso

### 4.1 GridSearchCV para encontrar el mejor alpha

In [ ]:
param_grid = {
    'alpha': list(np.arange(0.001, 0.5, 0.01)) + list(np.arange(0.5, 5.0, 0.1))
}

gs = GridSearchCV(
    Lasso(max_iter=10000, random_state=42),
    param_grid, cv=5, scoring='r2', n_jobs=-1
)
gs.fit(X_train_sc, y_train)

print(f'Mejor alpha (GridSearchCV): {gs.best_params_["alpha"]:.4f}')
print(f'R² en validación cruzada:   {gs.best_score_:.4f}')

### 4.2 Modelo optimizado — métricas finales

In [ ]:
best_alpha = gs.best_params_['alpha']

lasso_opt = Lasso(alpha=best_alpha, max_iter=10000, random_state=42)
lasso_opt.fit(X_train_sc, y_train)
y_pred_opt = lasso_opt.predict(X_test_sc)

mse_opt = mean_squared_error(y_test, y_pred_opt)
r2_opt  = r2_score(y_test, y_pred_opt)

print(f'--- Lasso optimizado (alpha={best_alpha:.4f}) ---')
print(f'MSE:  {mse_opt:,.2f}')
print(f'RMSE: {np.sqrt(mse_opt):,.2f}')
print(f'R²:   {r2_opt:.4f} ({r2_opt*100:.2f}%)')
print()
coefs = pd.Series(lasso_opt.coef_, index=X.columns)
print(f'Variables reducidas a 0: {(coefs == 0).sum()} de {len(coefs)}')

### 4.3 Coeficientes del modelo optimizado

In [ ]:
coefs_nz = coefs[coefs != 0].sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d73027' if v > 0 else '#4575b4' for v in coefs_nz.values]
coefs_nz.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Coeficientes — Lasso optimizado (alpha={best_alpha:.4f})')
ax.set_xlabel('Valor del coeficiente (datos normalizados)')
plt.tight_layout()
plt.show()

### 4.4 Real vs Predicho y distribución de residuos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_opt, alpha=0.4, color='crimson', s=15)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
axes[0].set_xlabel('Valores reales')
axes[0].set_ylabel('Valores predichos')
axes[0].set_title('Real vs Predicho — Lasso optimizado')

residuos = y_test - y_pred_opt
sns.histplot(residuos, kde=True, ax=axes[1], color='crimson')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Distribución de residuos')
axes[1].set_xlabel('Residuo (real - predicho)')

plt.tight_layout()
plt.show()

### 4.5 Comparación final de los tres modelos

In [ ]:
final = pd.DataFrame({
    'Modelo': ['LinearRegression', 'Lasso (alpha=1.0)', f'Lasso optimizado (alpha={best_alpha:.4f})'],
    'MSE':  [mse_lr, mse_ldef, mse_opt],
    'RMSE': [np.sqrt(mse_lr), np.sqrt(mse_ldef), np.sqrt(mse_opt)],
    'R²':   [r2_lr, r2_ldef, r2_opt]
})
print(final.to_string(index=False))
print()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(final['Modelo'], final['R²'], color=['steelblue','orange','crimson'], edgecolor='white')
for bar, val in zip(bars, final['R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1)
ax.set_ylabel('R²')
ax.set_title('Comparación final de modelos')
ax.tick_params(axis='x', rotation=10)
plt.tight_layout()
plt.show()

## Conclusiones

- **`Heart disease_number`** está altamente correlacionada con los conteos absolutos de población y otras condiciones de salud (COPD_number, diabetes_number). Esto es esperable: condados más poblados tienen más casos en términos absolutos.

- Las variables de **recursos sanitarios** (`Active Physicians per 100000`, `Total Hospitals`) muestran correlación positiva con el número de casos — no porque más médicos cause más enfermedad, sino porque los condados grandes tienen más de ambos.

- El **modelo de regresión lineal base** obtiene un R² alto gracias a estas correlaciones de volumen poblacional.

- **Lasso con alpha=1.0** penaliza fuertemente los coeficientes y puede dejar muy pocas variables activas. El alpha óptimo (encontrado por GridSearchCV) logra equilibrar regularización y rendimiento.

- La **normalización** fue esencial para que Lasso funcione correctamente, dado que las variables tienen escalas muy distintas (porcentajes vs. conteos absolutos de miles de personas).